# Palace to SAX: a differential travelling-wave MZM

This tutorial builds a circuit-ready MZM model in four steps:

1. simulate a four-port GSGSG transmission line in Palace;
2. use scikit-rf to convert single-ended ports to differential/common modes;
3. de-embed the wave-port discontinuities and attach junction RC loads in SAX;
4. cascade twenty 100 µm cells into a 2 mm travelling-wave MZM.

The important convention is the port order `[upper left, lower left, upper right, lower right]`. With that order, `Network.se2gmm(p=2)` produces `[differential left, differential right, common left, common right]`. The final six-port circuit contains three pairs, so `se2gmm(p=3)` directly produces the differential RF and EO responses.

The saved six-port Touchstone file lets the circuit sections run without launching Palace again.

**Requirements:**

- IHP PDK: `uv pip install ihp-gdsfactory`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation


### Define GSGSG electrode

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK

PDK.activate()


@gf.cell
def gsgsg_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    signal_pitch: float = 80,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    """
    Create a GSGSG (differential coplanar) electrode.

    The two signal electrodes sit on `signal_pitch` centres, separated by a
    shared central ground, so each line is its own CPW and the two are only
    weakly coupled.

    Args:
        length: horizontal length of the electrodes
        s_width: width of each signal electrode
        g_width: width of the two outer ground electrodes
        gap_width: gap between a signal electrode and its adjacent grounds
        signal_pitch: centre-to-centre spacing of the two signal electrodes
        layer: layer for the metal
    """
    c = gf.Component()

    center_g_width = signal_pitch - s_width - 2 * gap_width
    if center_g_width <= 0:
        raise ValueError(
            f"signal_pitch={signal_pitch} is too small for s_width={s_width} "
            f"and gap_width={gap_width}: the central ground would vanish."
        )

    s_center = signal_pitch / 2
    g_center = s_center + s_width / 2 + gap_width + g_width / 2

    # Central ground, shared by both lines
    c << gf.c.rectangle((length, center_g_width), centered=True, layer=layer)

    for sign in (+1, -1):
        sig = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)
        sig.move((0, sign * s_center))

        gnd = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
        gnd.move((0, sign * g_center))

    # Port order matters: scikit-rf se2gmm(p=2) treats ports (0, 1) as the
    # left-hand pair and (2, 3) as the right-hand pair, with 0-2 and 1-3 the
    # two through paths. So: o1/o3 = upper line, o2/o4 = lower line.
    for name, x, orientation, sign in (
        ("o1", -length / 2, 180, +1),
        ("o2", -length / 2, 180, -1),
        ("o3", length / 2, 0, +1),
        ("o4", length / 2, 0, -1),
    ):
        c.add_port(
            name=name,
            center=(x, sign * s_center),
            width=s_width,
            orientation=orientation,
            port_type="electrical",
            layer=layer,
        )

    c.info["s_width"] = s_width
    c.info["g_width"] = g_width
    c.info["gap_width"] = gap_width
    c.info["signal_pitch"] = signal_pitch
    c.info["center_g_width"] = center_g_width
    return c


c = gsgsg_electrode()
cc = c.copy()
cc.draw_ports()
cc

### Configure simulation

Each signal line gets its own wave port, so the port rectangles must **not** span the full boundary face — `max_size=True` would make the two ports on a face identical and overlapping. Two settings control the extent instead:

- **`lateral_margin`** sets the half-width of the port box around the signal centre. At 25 µm the box spans $y \in [5, 75]$ for the upper line, so its edges land ~10 µm inside the central ground and ~10 µm inside the outer ground, where the fields are already small. Palace treats the port cross-section boundary as PEC in the port eigenproblem, and terminating it inside a ground conductor is a good approximation of that. The two boxes on a face are left 10 µm apart.
- **`full_height=True`** makes the port span the simulation domain in $z$. This matters: the air box is not part of the layer stack, so without it the port box is clamped to the stack and collapses onto the conductor itself, putting a PEC lid directly above the electrodes and badly corrupting $Z_c$ and $n_{\mathrm{eff}}$.

**All four ports are excited.** Palace assigns one excitation index per excited port, and the results parser fills missing S-matrix entries only by reciprocity — never by geometric symmetry. Exciting fewer ports would leave whole rows of the 4×4 at exactly zero, and the mixed-mode transform needs the complete matrix. Four excitations is ~4× the solve cost of the single-ended notebook. The 10 MHz–50 GHz sweep includes a quasi-DC diagnostic point while retaining the microwave band; true DC behavior should come from an electrostatic or circuit model.

In [ ]:
from gsim.common.stack import get_stack
from gsim.palace import DrivenSim

PORT_NAMES = ("o1", "o2", "o3", "o4")


def setup_sim(cell, output_dir="./palace-sim-gsgsg-waveport"):
    sim = DrivenSim()
    sim.set_output_dir(output_dir)
    sim.set_geometry(cell)

    stack = get_stack()  # auto-detects active PDK
    sim.set_stack(stack)
    sim.set_airbox(margin_x=0.0, margin_y=50, z_above=100.0, z_below=100.0)

    # One wave port per signal line per face. Partial width (so the two ports
    # on a face do not overlap) but full height in z.
    lateral_margin = cell.info["gap_width"] + cell.info["g_width"] / 4

    for name in PORT_NAMES:
        sim.add_wave_port(
            name,
            layer="topmetal2",
            lateral_margin=lateral_margin,
            full_height=True,
            mode=1,
            excited=True,
        )

    # Include the quasi-DC RF limit. Wave ports do not define a true DC
    # solution, but 10 MHz is low enough to expose any fixture offset.
    sim.set_driven(fmin=10e6, fmax=50e9, num_points=201)

    print(sim.validate_config())

    return sim


sim = setup_sim(c)

### Generate mesh

In [ ]:
sim.mesh(preset="default", refined_mesh_size=2.0, max_mesh_size=40.0, fmax=60e9)

In [ ]:
# Confirm the four port boxes are partial-width, full-height and non-overlapping
for p in sim._last_mesh_result.port_info:
    print(
        f"P{p['portnumber']}  x={p['xmin']:7.1f}  "
        f"y=[{p['ymin']:6.1f}, {p['ymax']:6.1f}]  "
        f"z=[{p['zmin']:7.1f}, {p['zmax']:7.1f}]"
    )

In [ ]:
sim.plot_mesh(
    style="solid",
    transparent_groups=["air__None", "SiO2__None", "SiO2__passive", "air__passive"],
    interactive=True,
)

### Run simulation

In [ ]:
results = sim.run(check_cache=True)

In [ ]:
results.plot_interactive()

## 2. Convert the four-port result to mixed mode

`Network.se2gmm()` performs the single-ended-to-mixed-mode basis change and updates the reference impedances: a 50 Ω pair becomes 100 Ω differential and 25 Ω common mode. We keep the full mixed-mode network so mode conversion remains visible, and extract the differential and common two-ports only when calculating line parameters.

For a uniform reciprocal line,

$$Z_c=\sqrt{B/C}, \qquad
e^{\gamma \ell}=\frac{A+D}{2}+\frac{B}{Z_c}.$$

Choosing the positive-real branch of $Z_c$ and unwrapping the phase of $e^{\gamma\ell}$ avoids ambiguous ABCD eigenvalue tracking at $\beta\ell=\pi$.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf
from scipy.constants import speed_of_light
from skrf.calibration.deembedding import IEEEP370_SE_NZC_2xThru


def mixed_mode(network, pairs=2):
    """Return a mixed-mode copy; keep the single-ended input unchanged."""
    out = network.copy()
    out.se2gmm(p=pairs)
    return out


def extract_modal_parameters(net, length_m):
    """Extract Zc, propagation constant and effective index from a uniform 2-port."""
    a = net.a
    A, B, C, D = a[:, 0, 0], a[:, 0, 1], a[:, 1, 0], a[:, 1, 1]
    zc = np.sqrt(B / C)
    zc = np.where(np.real(zc) < 0, -zc, zc)
    exp_gamma_l = (A + D) / 2 + B / zc
    gamma = (
        np.log(np.abs(exp_gamma_l)) + 1j * np.unwrap(np.angle(exp_gamma_l))
    ) / length_m
    neff = np.imag(gamma) * speed_of_light / net.frequency.w
    return {"zc": zc, "gamma": gamma, "neff": neff}


def to_mixed_mode(result):
    """Order a Palace four-port and return its single-ended and modal Networks."""
    net = result.to_skrf()
    net.frequency.unit = "GHz"
    names = list(result.port_names)
    if set(names) == set(PORT_NAMES):
        net.renumber([names.index(name) for name in PORT_NAMES], range(4))
        net.port_names = list(PORT_NAMES)
    elif not all(name.startswith("p") and name[1:].isdigit() for name in names):
        raise ValueError(f"Cannot infer mixed-mode pairing from ports {names}.")

    mm = mixed_mode(net, pairs=2)
    return {
        "se": net,
        "mm": mm,
        "single": net.subnetwork([0, 2]),
        "diff": mm.subnetwork([0, 1]),
        "comm": mm.subnetwork([2, 3]),
    }


def p370_extract(short_2xthru, long_fix_dut_fix, length_m):
    """De-embed a short 2x-thru and extract the added uniform-line length."""
    split = IEEEP370_SE_NZC_2xThru(
        dummy_2xthru=short_2xthru,
        z0=float(np.real(np.median(short_2xthru.z0[:, 0]))),
        use_z_instead_ifft=True,
        name="wave-port 2x-thru",
    )
    dut = split.deembed(long_fix_dut_fix)
    return {"dut": dut, "split": split, **extract_modal_parameters(dut, length_m)}

In [ ]:
nets = to_mixed_mode(results)
line = {
    mode: extract_modal_parameters(nets[mode], length_m=800e-6)
    for mode in ("single", "diff", "comm")
}

for mode, values in line.items():
    print(
        f"{mode:7s}: Zc={np.median(values['zc'].real):6.2f} Ω, "
        f"neff={np.median(values['neff']):5.3f}"
    )

In [ ]:
rf.stylely()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for mode, style in (("single", "-"), ("diff", "--"), ("comm", ":")):
    frequency = nets[mode].frequency
    frequency.plot(line[mode]["zc"].real, style, ax=axes[0], label=mode)
    frequency.plot(line[mode]["neff"], style, ax=axes[1], label=mode)

axes[0].set(title="Characteristic impedance", ylabel=r"$Re(Z_c)$ [Ω]")
axes[1].set(title="Effective index", ylabel=r"$n_\mathrm{eff}$")
for axis in axes:
    axis.legend()
    axis.grid(True)
fig.tight_layout()

### Check mode conversion

Extracting separate differential and common two-ports assumes that conversion between them is small. The full mixed-mode `Network` makes that assumption easy to check. We also plot single-ended near-end crosstalk. `Frequency.plot()` supplies the frequency scaling and axis label, while `skrf.mathFunctions` handles complex-to-dB conversion.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
mode_conversion = np.maximum(
    rf.mathFunctions.complex_2_magnitude(nets["mm"].s[:, 0, 2]),
    rf.mathFunctions.complex_2_magnitude(nets["mm"].s[:, 2, 0]),
)
nets["mm"].frequency.plot(
    rf.mathFunctions.magnitude_2_db(mode_conversion),
    ax=axes[0],
    label=r"max($S_{dc}, S_{cd}$)",
)
nets["se"].frequency.plot(
    rf.mathFunctions.complex_2_db(nets["se"].s[:, 1, 0]),
    ax=axes[1],
    label="near-end crosstalk",
)
axes[0].set(title="Mode conversion", ylabel="Magnitude [dB]")
axes[1].set(title="Line-to-line coupling", ylabel="Magnitude [dB]")
for axis in axes:
    axis.legend()
    axis.grid(True)
fig.tight_layout()

## 3. Calibrate the loaded unit-cell reference planes

The MZM unit cell uses one T-bar pair per arm every 100 µm. A 100 µm and a 400 µm loaded line provide an IEEE P370 non-zero-length 2x-thru calibration. De-embedding the short result from the long one leaves a 300 µm line from which we obtain the differential and common propagation constants. Those line parameters let us remove the 50 µm of physical line contained in each P370 half-fixture, leaving only the wave-port discontinuity.

This calibration is only needed when regenerating the Touchstone model. To work on the SAX circuit, skip to the saved-data section.


In [ ]:
@gf.cell
def gsgsg_tbar_electrode(
    length: float = 100,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    signal_pitch: float = 80,
    tbar_period: float = 100,
    tbar_stem_width: float = 4,
    tbar_arm_length: float = 95,
    tbar_arm_width: float = 2,
    tbar_gap: float = 3,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    """GSGSG electrode with one facing T-bar pair per period and arm."""
    n_periods = round(length / tbar_period)
    if not np.isclose(n_periods * tbar_period, length):
        raise ValueError("tbar_period must divide length exactly.")

    c = gf.Component()
    base = c << gsgsg_electrode(
        length=length,
        s_width=s_width,
        g_width=g_width,
        gap_width=gap_width,
        signal_pitch=signal_pitch,
        layer=layer,
    )
    c.add_ports(base.ports)

    stem_length = (gap_width - tbar_gap) / 2 - tbar_arm_width
    if stem_length <= 0:
        raise ValueError("The T-bar dimensions leave no room for the stem.")
    signal_inner_edge = signal_pitch / 2 - s_width / 2
    ground_inner_edge = signal_inner_edge - gap_width
    x_centers = -length / 2 + (np.arange(n_periods) + 0.5) * tbar_period

    for sign in (+1, -1):
        for x0 in x_centers:
            for y_root, direction in (
                (signal_inner_edge, -1),
                (ground_inner_edge, +1),
            ):
                stem = c << gf.c.rectangle(
                    (tbar_stem_width, stem_length), centered=True, layer=layer
                )
                stem.move((x0, sign * (y_root + direction * stem_length / 2)))
                arm = c << gf.c.rectangle(
                    (tbar_arm_length, tbar_arm_width), centered=True, layer=layer
                )
                arm.move(
                    (
                        x0,
                        sign
                        * (y_root + direction * (stem_length + tbar_arm_width / 2)),
                    )
                )

    for key, value in {
        "s_width": s_width,
        "g_width": g_width,
        "gap_width": gap_width,
        "signal_pitch": signal_pitch,
        "tbar_period": tbar_period,
        "tbar_arm_length": tbar_arm_length,
        "tbar_gap": tbar_gap,
        "n_periods": n_periods,
    }.items():
        c.info[key] = value
    return c


gsgsg_tbar_electrode()

In [ ]:
tbar_lines = [gsgsg_tbar_electrode(length=length) for length in (100, 400)]
tbar_jobs = []
for cell in tbar_lines:
    loaded_sim = setup_sim(cell, output_dir="./palace-sim-gsgsg-tbar")
    loaded_sim.mesh(
        preset="default",
        refined_mesh_size=2.0,
        max_mesh_size=40.0,
        fmax=60e9,
        auto_size=True,
    )
    tbar_jobs.append(loaded_sim.run(wait=False, check_cache=True))

In [ ]:
import gsim

tbar_results = gsim.wait_for_results(tbar_jobs)

In [ ]:
tbar_networks = [to_mixed_mode(result) for result in tbar_results]
tbar_deembedded = {
    mode: {
        300: p370_extract(
            tbar_networks[0][mode], tbar_networks[1][mode], length_m=300e-6
        )
    }
    for mode in ("diff", "comm")
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for mode, style in (("diff", "-"), ("comm", "--")):
    result = tbar_deembedded[mode][300]
    frequency = result["dut"].frequency
    frequency.plot(result["zc"].real, style, ax=axes[0], label=mode)
    frequency.plot(result["neff"], style, ax=axes[1], label=mode)
axes[0].set(title="Loaded-line impedance", ylabel=r"$Re(Z_c)$ [Ω]")
axes[1].set(title="Loaded-line effective index", ylabel=r"$n_\mathrm{eff}$")
for axis in axes:
    axis.legend()
    axis.grid(True)
fig.tight_layout()

## 4. Simulate one circuit-ready MZM cell

The 100 µm cell below adds one in-plane lumped port across each arm's T-bar gap. Palace therefore returns six ports: four travelling-wave terminals and two junction terminals. Every port is excited so the exported S-matrix contains all coupling terms.

The P370 fixture includes a wave-port discontinuity plus 50 µm of line on each side. We synthesize that line from the extracted $Z_c$ and $\gamma$, remove it, convert the remaining differential/common error boxes back with `gmm2se()`, and de-embed only the four boundary ports. The local junction ports remain untouched.


In [ ]:
@gf.cell
def gsgsg_tbar_lumped_segment(
    length: float = 100.0,
    tbar_period: float = 100.0,
    s_width: float = 20.0,
    g_width: float = 40.0,
    gap_width: float = 15.0,
    signal_pitch: float = 80.0,
    tbar_stem_width: float = 4.0,
    tbar_arm_length: float = 95.0,
    tbar_arm_width: float = 2.0,
    tbar_gap: float = 3.0,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    """T-bar-loaded GSGSG section with a gap port at every T-bar pair."""
    n_periods = round(length / tbar_period)
    if not np.isclose(n_periods * tbar_period, length):
        raise ValueError(f"tbar_period={tbar_period} must divide length={length}.")

    c = gf.Component()
    base = c << gsgsg_tbar_electrode(
        length=length,
        s_width=s_width,
        g_width=g_width,
        gap_width=gap_width,
        signal_pitch=signal_pitch,
        tbar_period=tbar_period,
        tbar_stem_width=tbar_stem_width,
        tbar_arm_length=tbar_arm_length,
        tbar_arm_width=tbar_arm_width,
        tbar_gap=tbar_gap,
        layer=layer,
    )
    c.add_ports(base.ports)

    stem_length = (gap_width - tbar_gap) / 2 - tbar_arm_width
    s_inner = signal_pitch / 2 - s_width / 2
    gap_center_from_axis = s_inner - stem_length - tbar_arm_width - tbar_gap / 2
    x_centers = [-length / 2 + (i + 0.5) * tbar_period for i in range(n_periods)]

    for arm, sign, orientation in (
        ("upper", +1, 270),
        ("lower", -1, 90),
    ):
        for i, x0 in enumerate(x_centers, start=1):
            c.add_port(
                name=f"{arm}_tbar_{i}",
                center=(x0, sign * gap_center_from_axis),
                width=tbar_arm_length,
                orientation=orientation,
                port_type="electrical",
                layer=layer,
            )

    for key, value in (
        ("s_width", s_width),
        ("g_width", g_width),
        ("gap_width", gap_width),
        ("signal_pitch", signal_pitch),
        ("tbar_period", tbar_period),
        ("tbar_arm_length", tbar_arm_length),
        ("tbar_gap", tbar_gap),
        ("n_periods", n_periods),
    ):
        c.info[key] = value
    return c


mzm_segment = gsgsg_tbar_lumped_segment()
TBAR_LUMPED_PORT_NAMES = tuple(
    f"{arm}_tbar_{i}"
    for arm in ("upper", "lower")
    for i in range(1, int(mzm_segment.info["n_periods"]) + 1)
)
mzm_segment_display = mzm_segment.copy()
mzm_segment_display.draw_ports()
mzm_segment_display

In [ ]:
def setup_mzm_segment_sim(cell):
    sim = setup_sim(cell, output_dir="./palace-sim-gsgsg-tbar-lumped-segment")
    for name in TBAR_LUMPED_PORT_NAMES:
        sim.add_port(
            name,
            geometry="inplane",
            layer="topmetal2",
            length=cell.info["tbar_gap"],
            impedance=50.0,
            excited=True,
        )
    print(sim.validate_config())
    return sim


mzm_sim = setup_mzm_segment_sim(mzm_segment)
mzm_sim.mesh(preset="default", refined_mesh_size=2.0, max_mesh_size=40.0, fmax=60e9)
print(
    f"ports={len(mzm_sim._last_mesh_result.port_info)}  "
    f"min quality={mzm_sim._last_mesh_result.mesh_stats['quality']['min']:.3f}"
)

In [ ]:
job_id = mzm_sim.run(check_cache=True, wait=False)

In [ ]:
mzm_raw = gsim.wait_for_results(job_id)

In [ ]:
def uniform_line_network(template, params, length_m, name):
    """Create an extracted uniform line with scikit-rf's media model."""
    medium = rf.media.DefinedGammaZ0(
        frequency=template.frequency,
        z0_port=template.z0[:, 0],
        z0=params["zc"],
        gamma=params["gamma"],
    )
    return medium.line(length_m, unit="m", name=name)


def port_only_error_boxes(p370_result, thru_length_m=100e-6):
    """Remove half the physical thru from each P370 error box."""
    half_line = uniform_line_network(
        p370_result["dut"], p370_result, thru_length_m / 2, "half loaded cell"
    )
    split = p370_result["split"]
    return split.s_side1**half_line.inv, half_line.inv ** split.s_side2.flipped()


def modal_pair_to_single_ended(diff_error, comm_error, name):
    """Combine differential/common two-ports and return a four-port fixture."""
    s = np.zeros((len(diff_error), 4, 4), dtype=complex)
    s[:, :2, :2] = diff_error.s
    s[:, 2:, 2:] = comm_error.s
    fixture = rf.Network(
        frequency=diff_error.frequency,
        s=s,
        z0=np.concatenate((diff_error.z0, comm_error.z0), axis=1),
        name=name,
    )
    fixture.gmm2se(p=2)
    return fixture


def deembed_boundary_ports(net, left_error, right_error):
    """Remove the two coupled wave-port fixtures while retaining local ports."""
    left_inverse = left_error.inv
    left_inverse.port_names = ["o1", "o2", "_left_upper", "_left_lower"]
    out = rf.connect(left_inverse, 2, net, list(net.port_names).index("o1"), num=2)

    right_inverse = right_error.inv
    right_inverse.port_names = ["_right_upper", "_right_lower", "o3", "o4"]
    out = rf.connect(out, list(out.port_names).index("o3"), right_inverse, 0, num=2)

    desired = list(PORT_NAMES) + list(TBAR_LUMPED_PORT_NAMES)
    current = list(out.port_names)
    if set(current) != set(desired):
        raise ValueError(f"Expected ports {desired}, got {current}.")
    out.renumber([current.index(name) for name in desired], range(len(desired)))
    out.port_names = desired
    out.name = "100 um T-bar MZM cell, wave ports de-embedded"
    return out


port_errors = {
    mode: port_only_error_boxes(tbar_deembedded[mode][300]) for mode in ("diff", "comm")
}
left_port_error = modal_pair_to_single_ended(
    port_errors["diff"][0], port_errors["comm"][0], "left wave-port fixture"
)
right_port_error = modal_pair_to_single_ended(
    port_errors["diff"][1], port_errors["comm"][1], "right wave-port fixture"
)

In [ ]:
mzm_raw_network = mzm_raw.to_skrf()
mzm_raw_network.frequency.unit = "GHz"
mzm_deembedded = deembed_boundary_ports(
    mzm_raw_network, left_port_error, right_port_error
)

print(f"{mzm_deembedded.nports}-port network: {mzm_deembedded.port_names}")
mzm_deembedded.frequency.plot(
    rf.mathFunctions.complex_2_db(mzm_deembedded.s[:, 2, 0]),
    label=r"$S(o3,o1)$",
)
plt.ylabel("Magnitude [dB]")
plt.grid(True)
plt.legend()

The resulting `mzm_deembedded` network is the reusable electromagnetic unit cell. Export it once; the remaining tutorial loads the Touchstone data and does not call Palace. The RF cell pitch is 100 µm, while the junction-active T-bar length is 95 µm; the RC model below keeps those lengths separate.


In [ ]:
mzm_length_um = float(mzm_segment.info["n_periods"]) * float(
    mzm_segment.info["tbar_period"]
)
mzm_active_length_um = float(mzm_segment.info["n_periods"]) * float(
    mzm_segment.info["tbar_arm_length"]
)
mzm_deembedded.write_touchstone(
    f"mzm_tbar_{mzm_length_um:g}um_tbar{mzm_active_length_um:g}um_"
    f"{mzm_deembedded.nports}port_deembedded",
    form="ri",
    write_z0=True,
)

## 5. Load the saved cell and add the junction RC model

Each local gap port is terminated by

$$Z_\mathrm{load}(f)=R_s+\left(R_j^{-1}+j2\pi fC_j\right)^{-1}.$$

SAX performs the connection, while an independent S-matrix block elimination verifies it. Resistance scales inversely and capacitance directly with the 95 µm active T-bar length; the 100 µm pitch is retained separately for RF propagation and cascade length. The example bias table is inherited from the circuit proposal and is not an IHP calibration. The isolated non-passive 10 MHz de-embedding sample is excluded; the retained sweep starts at 259.95 MHz.


In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import sax

jax.config.update("jax_enable_x64", True)

RC_Z0 = 50.0
RC_RF_PORTS = ("o1", "o2", "o3", "o4")
RC_TOUCHSTONE_PATH = Path("mzm_tbar_100um_tbar95um_6port_deembedded.s6p")
if "mzm_deembedded" in globals():
    rc_em = mzm_deembedded.copy()
elif RC_TOUCHSTONE_PATH.exists():
    rc_em = rf.Network(RC_TOUCHSTONE_PATH)
else:
    raise RuntimeError(
        f"Finish the 95 µm T-bar simulation and export {RC_TOUCHSTONE_PATH}."
    )
rc_em.frequency.unit = "GHz"
rc_em.renormalize(RC_Z0)

valid = rc_em.f >= 100e6
if not np.all(valid):
    print("Dropped samples [GHz]:", rc_em.frequency.f_scaled[~valid])
rc_em = rc_em[np.flatnonzero(valid)]
if np.linalg.svd(rc_em.s, compute_uv=False)[:, 0].max() > 1.001:
    raise ValueError("The retained de-embedded sweep is materially non-passive.")

rc_arm_ports = {
    arm: sorted(name for name in rc_em.port_names if name.startswith(f"{arm}_tbar_"))
    for arm in ("upper", "lower")
}
rc_local_ports = rc_arm_ports["upper"] + rc_arm_ports["lower"]
if set(rc_em.port_names) != set(RC_RF_PORTS) | set(rc_local_ports):
    raise ValueError(f"Unexpected Touchstone ports: {rc_em.port_names}")

RC_CELL_LENGTH_M = 100e-6
RC_ACTIVE_LENGTH_M = 95e-6
if "mzm_segment" in globals():
    simulated_cell_length = (
        float(mzm_segment.info["n_periods"])
        * float(mzm_segment.info["tbar_period"])
        * 1e-6
    )
    simulated_active_length = (
        float(mzm_segment.info["n_periods"])
        * float(mzm_segment.info["tbar_arm_length"])
        * 1e-6
    )
    if not np.isclose(simulated_cell_length, RC_CELL_LENGTH_M):
        raise ValueError("Touchstone and geometry cell lengths do not agree.")
    if not np.isclose(simulated_active_length, RC_ACTIVE_LENGTH_M):
        raise ValueError("Touchstone and geometry active lengths do not agree.")
RC_BIAS_V = -1
RC_BIAS_TABLE = {
    0: {"r_dc": 10000.0, "cj_per_m": 0.85 * 250e-12},
    -1: {"r_dc": 10000.0, "cj_per_m": 0.85 * 225e-12},
    -2: {"r_dc": 10000.0, "cj_per_m": 0.85 * 210e-12},
    -3: {"r_dc": 10000.0, "cj_per_m": 0.85 * 190e-12},
}
RC_REFERENCE_LENGTH_M = 100e-6
rc_rs_length = 8.9e-3
rc_cfg = RC_BIAS_TABLE[RC_BIAS_V]
rc_rj_length = (
    rc_cfg["r_dc"] - rc_rs_length / RC_REFERENCE_LENGTH_M
) * RC_REFERENCE_LENGTH_M
rc_settings = {}
for ports in rc_arm_ports.values():
    site_length = RC_ACTIVE_LENGTH_M / len(ports)
    for port in ports:
        rc_settings[port] = {
            "rs": rc_rs_length / site_length,
            "rj": rc_rj_length / site_length,
            "cj": rc_cfg["cj_per_m"] * site_length,
        }

print(
    f"Cell pitch={RC_CELL_LENGTH_M * 1e6:g} µm; "
    f"active length={RC_ACTIVE_LENGTH_M * 1e6:g} µm; "
    f"junction sites={rc_arm_ports}; values={rc_settings[rc_local_ports[0]]}"
)

In [ ]:
def rc_series_resistor(*, r=50.0):
    reflection = r / (r + 2 * RC_Z0)
    transmission = 2 * RC_Z0 / (r + 2 * RC_Z0)
    return {
        ("a", "a"): reflection,
        ("b", "b"): reflection,
        ("a", "b"): transmission,
        ("b", "a"): transmission,
    }


def rc_junction(*, frequency=10e6, rj=1e4, cj=1e-15):
    admittance = 1 / rj + 2j * jnp.pi * jnp.asarray(frequency) * cj
    return {("p", "p"): (1 - RC_Z0 * admittance) / (1 + RC_Z0 * admittance)}


def skrf_to_sax(network):
    """Interpolate a calibrated scikit-rf Network as a SAX model."""
    frequency = jnp.asarray(network.f)
    scattering = jnp.asarray(network.s)
    ports = tuple(network.port_names)

    def model(*, frequency=frequency[0]):
        return {
            (out_port, in_port): (
                jnp.interp(frequency, network.f, scattering[:, i, j].real)
                + 1j * jnp.interp(frequency, network.f, scattering[:, i, j].imag)
            )
            for i, out_port in enumerate(ports)
            for j, in_port in enumerate(ports)
        }

    return model


def sax_to_skrf(sdict, ports, frequency, z0=RC_Z0, name=None):
    """Pack a frequency-vectorized SAX dictionary into a scikit-rf Network."""
    s = np.stack(
        [
            np.stack([np.asarray(sdict[out, inp]) for inp in ports], axis=-1)
            for out in ports
        ],
        axis=1,
    )
    network = rf.Network(frequency=frequency.copy(), s=s, z0=z0, name=name)
    network.port_names = list(ports)
    return network


instances = {"palace": {"component": "palace"}}
connections = {}
for port, values in rc_settings.items():
    instances[f"rs_{port}"] = {"component": "rs", "settings": {"r": values["rs"]}}
    instances[f"junction_{port}"] = {
        "component": "junction",
        "settings": {"rj": values["rj"], "cj": values["cj"]},
    }
    connections[f"palace,{port}"] = f"rs_{port},a"
    connections[f"rs_{port},b"] = f"junction_{port},p"

rc_circuit, _ = sax.circuit(
    netlist={
        "instances": instances,
        "connections": connections,
        "ports": {port: f"palace,{port}" for port in RC_RF_PORTS},
    },
    models={
        "palace": skrf_to_sax(rc_em),
        "rs": rc_series_resistor,
        "junction": rc_junction,
    },
)
rc_sdict = jax.block_until_ready(rc_circuit(frequency=jnp.asarray(rc_em.f)))
mzm_rc_loaded = sax_to_skrf(
    rc_sdict, RC_RF_PORTS, rc_em.frequency, name="RC-loaded Palace cell"
)

### Verify the load connection in mixed mode

The block termination

$$S_\mathrm{loaded}=S_{ee}+S_{ei}\Gamma(I-S_{ii}\Gamma)^{-1}S_{ie}$$

provides an implementation-independent check of the SAX circuit. We then use `se2gmm(p=2)` to compare open and loaded differential responses. A matched differential termination is half the extracted differential impedance on each single-ended output port.


In [ ]:
def terminate_local_ports(net, gamma):
    external = [net.port_names.index(port) for port in RC_RF_PORTS]
    internal = [net.port_names.index(port) for port in rc_local_ports]
    s = net.s[:, external + internal][:, :, external + internal]
    see, sei = s[:, :4, :4], s[:, :4, 4:]
    sie, sii = s[:, 4:, :4], s[:, 4:, 4:]
    feedback = np.linalg.solve(
        np.eye(len(internal))[None] - sii * gamma[:, None, :], sie
    )
    return see + (sei * gamma[:, None, :]) @ feedback


loads = np.column_stack(
    [
        values["rs"] + 1 / (1 / values["rj"] + 2j * np.pi * rc_em.f * values["cj"])
        for values in (rc_settings[port] for port in rc_local_ports)
    ]
)
gamma_load = (loads - RC_Z0) / (loads + RC_Z0)
np.testing.assert_allclose(
    mzm_rc_loaded.s, terminate_local_ports(rc_em, gamma_load), rtol=1e-8, atol=1e-10
)

mzm_rc_open = rf.Network(
    frequency=rc_em.frequency,
    s=terminate_local_ports(rc_em, np.ones_like(gamma_load)),
    z0=RC_Z0,
    name="open junction ports",
)
mzm_rc_open.port_names = list(RC_RF_PORTS)

loaded_mm = mixed_mode(mzm_rc_loaded, pairs=2)
loaded_diff = loaded_mm.subnetwork([0, 1])
loaded_line = extract_modal_parameters(loaded_diff, RC_CELL_LENGTH_M)
MZM_LOADED_ZDIFF = float(np.median(loaded_line["zc"].real))
MZM_RF_TERMINATION_SE = MZM_LOADED_ZDIFF / 2
print(
    f"Loaded Zdiff={MZM_LOADED_ZDIFF:.2f} Ω; "
    f"far-end termination={MZM_RF_TERMINATION_SE:.2f} Ω per conductor"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for network, style in ((mzm_rc_open, "--"), (mzm_rc_loaded, "-")):
    mm = mixed_mode(network, pairs=2)
    mm.frequency.plot(
        rf.mathFunctions.complex_2_db(mm.s[:, 1, 0]),
        style,
        ax=axes[0],
        label=network.name,
    )
    mm.frequency.plot(
        rf.mathFunctions.complex_2_db(mm.s[:, 0, 0]),
        style,
        ax=axes[1],
        label=network.name,
    )
axes[0].set(title="Differential transmission", ylabel=r"$S_{dd21}$ [dB]")
axes[1].set(title="Differential return loss", ylabel=r"$S_{dd11}$ [dB]")
for axis in axes:
    axis.legend()
    axis.grid(True)
fig.tight_layout()

## 6. Cascade twenty cells into a 2 mm MZM

Each SAX unit cell contains the six-port Palace model, two junction RC loads, and ideal high-impedance voltage probes. The probes accumulate the junction voltage along an optical path with group delay. Twenty identical 100 µm cells form the 2 mm device.

The final SAX dictionary is converted back to a six-port scikit-rf `Network`. Renormalizing only the two far RF ports expresses the extracted line match. Finally, `se2gmm(p=3)` converts the RF-input, RF-output, and EO-output pairs together; the desired responses are then ordinary mixed-mode S-parameters rather than hand-written wave combinations.


In [ ]:
C_LIGHT = 299_792_458.0
MZM_TOTAL_LENGTH_M = 2e-3
MZM_GROUP_INDEX = 3.97
MZM_CELL_LENGTH_M = RC_CELL_LENGTH_M
MZM_N_CELLS = round(MZM_TOTAL_LENGTH_M / MZM_CELL_LENGTH_M)
MZM_SITES_PER_ARM = len(rc_arm_ports["upper"])
MZM_SITE_LENGTH_M = MZM_CELL_LENGTH_M / MZM_SITES_PER_ARM
MZM_VCVS_GAIN = 1 / (MZM_N_CELLS * MZM_SITES_PER_ARM)


def voltage_probe_vcvs(*, gain=1.0):
    """High-impedance differential voltage probe with a through accumulator."""
    return sax.sdict(
        {
            ("sense_p", "sense_p"): 1.0,
            ("sense_n", "sense_n"): 1.0,
            ("out", "sense_p"): gain,
            ("out", "sense_n"): -gain,
            ("out", "acc_in"): 1.0,
            ("acc_in", "sense_p"): -gain,
            ("acc_in", "sense_n"): gain,
            ("acc_in", "out"): 1.0,
        }
    )


def optical_group_delay(
    *, frequency=10e6, length=MZM_SITE_LENGTH_M, ng=MZM_GROUP_INDEX
):
    phase = jnp.exp(-2j * jnp.pi * jnp.asarray(frequency) * length * ng / C_LIGHT)
    return {("a", "b"): phase, ("b", "a"): phase}


def voltage_buffer(*, gain=1.0, zi=50.0, zo=0.0):
    return {
        ("in", "in"): (zi - RC_Z0) / (zi + RC_Z0),
        ("out", "in"): 2 * gain * zi * RC_Z0 / ((zi + RC_Z0) * (zo + RC_Z0)),
        ("out", "out"): (zo - RC_Z0) / (zo + RC_Z0),
    }


def ideal_short():
    return {("p", "p"): -1.0 + 0j}


def ideal_splitter(num_ports):
    ports = tuple(f"p{i}" for i in range(1, num_ports + 1))

    def model():
        return {
            (out, inp): (2 - num_ports) / num_ports if out == inp else 2 / num_ports
            for out in ports
            for inp in ports
        }

    return model


print(f"{MZM_N_CELLS} x {MZM_CELL_LENGTH_M * 1e6:g} µm = 2 mm")

In [ ]:
def build_palace_eo_unit_cell():
    """Build one loaded Palace cell with two voltage-accumulator paths."""
    ground_ports = 1 + len(rc_local_ports)
    instances = {
        "palace": {"component": "palace"},
        "junction_ground": {"component": "short"},
        "ground_fanout": {"component": "ground_fanout"},
    }
    connections = {"junction_ground,p": "ground_fanout,p1"}
    last_delay = {}
    ground_index = 2

    for arm in ("upper", "lower"):
        instances[f"accumulator_input_{arm}"] = {
            "component": "buffer",
            "settings": {"gain": 1.0, "zi": RC_Z0, "zo": 0.0},
        }
        for site_index, palace_port in enumerate(rc_arm_ports[arm], start=1):
            tag = f"{arm}_{site_index}"
            values = rc_settings[palace_port]
            instances[f"rs_{tag}"] = {
                "component": "rs",
                "settings": {"r": values["rs"]},
            }
            instances[f"junction_{tag}"] = {
                "component": "junction",
                "settings": {"rj": values["rj"], "cj": values["cj"]},
            }
            instances[f"node_{tag}"] = {"component": "tee3"}
            instances[f"probe_{tag}"] = {
                "component": "probe",
                "settings": {"gain": MZM_VCVS_GAIN},
            }
            instances[f"probe_output_{tag}"] = {
                "component": "buffer",
                "settings": {"gain": 2.0, "zi": RC_Z0, "zo": RC_Z0},
            }
            instances[f"delay_{tag}"] = {
                "component": "delay",
                "settings": {"length": MZM_SITE_LENGTH_M, "ng": MZM_GROUP_INDEX},
            }

            connections[f"palace,{palace_port}"] = f"rs_{tag},a"
            connections[f"rs_{tag},b"] = f"node_{tag},p1"
            connections[f"junction_{tag},p"] = f"node_{tag},p2"
            connections[f"probe_{tag},sense_p"] = f"node_{tag},p3"
            connections[f"probe_{tag},sense_n"] = f"ground_fanout,p{ground_index}"
            ground_index += 1

            if site_index == 1:
                connections[f"accumulator_input_{arm},out"] = f"probe_{tag},acc_in"
            else:
                previous_tag = f"{arm}_{site_index - 1}"
                instances[f"accumulator_buffer_{tag}"] = {
                    "component": "buffer",
                    "settings": {"gain": 1.0, "zi": RC_Z0, "zo": 0.0},
                }
                connections[f"delay_{previous_tag},b"] = f"accumulator_buffer_{tag},in"
                connections[f"accumulator_buffer_{tag},out"] = f"probe_{tag},acc_in"

            connections[f"probe_{tag},out"] = f"probe_output_{tag},in"
            connections[f"probe_output_{tag},out"] = f"delay_{tag},a"
            last_delay[arm] = f"delay_{tag},b"

    ports = {
        "rf_in_upper": "palace,o1",
        "rf_in_lower": "palace,o2",
        "rf_out_upper": "palace,o3",
        "rf_out_lower": "palace,o4",
        "opt_in_upper": "accumulator_input_upper,in",
        "opt_in_lower": "accumulator_input_lower,in",
        "opt_out_upper": last_delay["upper"],
        "opt_out_lower": last_delay["lower"],
    }
    models = {
        "palace": skrf_to_sax(rc_em),
        "rs": rc_series_resistor,
        "junction": rc_junction,
        "tee3": ideal_splitter(3),
        "ground_fanout": ideal_splitter(ground_ports),
        "short": ideal_short,
        "probe": voltage_probe_vcvs,
        "buffer": voltage_buffer,
        "delay": optical_group_delay,
    }
    return sax.circuit(
        netlist={"instances": instances, "connections": connections, "ports": ports},
        models=models,
    )[0]


palace_eo_unit_cell = build_palace_eo_unit_cell()
unit_cell_s = jax.block_until_ready(palace_eo_unit_cell(frequency=jnp.asarray(rc_em.f)))
print(f"Palace EO unit cell: {len({p for pair in unit_cell_s for p in pair})} ports")

In [ ]:
def cascade_sax_cell(*, count, cell, port_connections):
    instances = {f"cell_{i}": {"component": "cell"} for i in range(count)}
    connections = {
        f"cell_{i},{out}": f"cell_{i + 1},{inp}"
        for i in range(count - 1)
        for out, inp in port_connections.items()
    }
    ports = {
        **{inp: f"cell_0,{inp}" for inp in port_connections.values()},
        **{out: f"cell_{count - 1},{out}" for out in port_connections},
    }
    return sax.circuit(
        netlist={"instances": instances, "connections": connections, "ports": ports},
        models={"cell": cell},
    )[0]


palace_eo_cascade = cascade_sax_cell(
    count=MZM_N_CELLS,
    cell=palace_eo_unit_cell,
    port_connections={
        "rf_out_upper": "rf_in_upper",
        "rf_out_lower": "rf_in_lower",
        "opt_out_upper": "opt_in_upper",
        "opt_out_lower": "opt_in_lower",
    },
)

mzm_2mm_netlist = {
    "instances": {
        "cascade": {"component": "cascade"},
        "optical_ground": {"component": "short"},
        "optical_ground_fanout": {"component": "tee3"},
        "eo_upper": {"component": "buffer", "settings": {"zi": RC_Z0, "zo": 0.0}},
        "eo_lower": {"component": "buffer", "settings": {"zi": RC_Z0, "zo": 0.0}},
    },
    "connections": {
        "optical_ground,p": "optical_ground_fanout,p1",
        "optical_ground_fanout,p2": "cascade,opt_in_upper",
        "optical_ground_fanout,p3": "cascade,opt_in_lower",
        "cascade,opt_out_upper": "eo_upper,in",
        "cascade,opt_out_lower": "eo_lower,in",
    },
    "ports": {
        "rf_in_upper": "cascade,rf_in_upper",
        "rf_in_lower": "cascade,rf_in_lower",
        "rf_out_upper": "cascade,rf_out_upper",
        "rf_out_lower": "cascade,rf_out_lower",
        "eo_upper": "eo_upper,out",
        "eo_lower": "eo_lower,out",
    },
}
palace_mzm_2mm, _ = sax.circuit(
    netlist=mzm_2mm_netlist,
    models={
        "cascade": palace_eo_cascade,
        "short": ideal_short,
        "tee3": ideal_splitter(3),
        "buffer": voltage_buffer,
    },
)

MZM_MODEL_PORTS = (
    "rf_in_upper",
    "rf_in_lower",
    "rf_out_upper",
    "rf_out_lower",
    "eo_upper",
    "eo_lower",
)
mzm_sdict = jax.block_until_ready(palace_mzm_2mm(frequency=jnp.asarray(rc_em.f)))
mzm_2mm_50ohm = sax_to_skrf(
    mzm_sdict, MZM_MODEL_PORTS, rc_em.frequency, name="2 mm MZM, 50 Ω outputs"
)
mzm_2mm_matched = mzm_2mm_50ohm.copy()
mzm_2mm_matched.renormalize(
    [RC_Z0, RC_Z0, MZM_RF_TERMINATION_SE, MZM_RF_TERMINATION_SE, RC_Z0, RC_Z0]
)
mzm_2mm_matched.name = "2 mm MZM, matched RF output"

In [ ]:
mzm_mm = mixed_mode(mzm_2mm_matched, pairs=3)
mzm_mm_50ohm = mixed_mode(mzm_2mm_50ohm, pairs=3)

# se2gmm(p=3) order: d_in, d_out, d_eo, c_in, c_out, c_eo.
eo_db = rf.mathFunctions.complex_2_db(mzm_mm.s[:, 2, 0])
eo_db -= eo_db[0]
eo_50ohm_db = rf.mathFunctions.complex_2_db(mzm_mm_50ohm.s[:, 2, 0])
eo_50ohm_db -= eo_50ohm_db[0]
rf_through_db = rf.mathFunctions.complex_2_db(mzm_mm.s[:, 1, 0])
rf_return_db = rf.mathFunctions.complex_2_db(mzm_mm.s[:, 0, 0])

eo_crossing = np.flatnonzero(eo_db <= -3)
rf_crossing = np.flatnonzero(rf_through_db <= -6.4)
eo_bandwidth = mzm_mm.frequency.f_scaled[eo_crossing[0]] if len(eo_crossing) else None
rf_bandwidth = mzm_mm.frequency.f_scaled[rf_crossing[0]] if len(rf_crossing) else None

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
mzm_mm.frequency.plot(eo_db, ax=axes[0], label="matched output")
mzm_mm_50ohm.frequency.plot(eo_50ohm_db, "--", ax=axes[0], label="50 Ω per conductor")
mzm_mm.frequency.plot(rf_through_db, ax=axes[1], label=r"$S_{dd21}$")
mzm_mm.frequency.plot(rf_return_db, ":", ax=axes[1], label=r"$S_{dd11}$")
axes[0].axhline(-3, color="tab:red", linestyle=":")
axes[1].axhline(-6.4, color="tab:red", linestyle=":")
axes[0].set(title="Differential EO response", ylabel="Normalized magnitude [dB]")
axes[1].set(title="Differential RF response", ylabel="Magnitude [dB]")
for axis in axes:
    axis.legend()
    axis.grid(True)
fig.suptitle("Palace + SAX travelling-wave MZM: 20 x 100 µm cells")
fig.tight_layout()

print(f"EO -3 dB bandwidth: {eo_bandwidth:.2f} GHz")
print(f"RF -6.4 dB bandwidth: {rf_bandwidth:.2f} GHz")